In [ ]:
from pathlib import Path

from astropy.io import fits
from astropy.io.fits.fitsrec import FITS_rec
from numpy.typing import NDArray
import numpy as np
from scipy.interpolate import griddata

import darksun as ds

ds.show.set_figures_darkbkg()

In [2]:
def load_fits_data(path: Path, ext: int = 1) -> FITS_rec:
    """Loads the FITS file data from chosen extension."""
    return fits.getdata(path, ext=ext, header=False)

def extract_transmission(
    data: FITS_rec,
    energy_range: tuple[float, float] = (2.0, 50.0),
) -> tuple[NDArray, NDArray]:
    """
    Extracts photons transmission values in specified `energy_range`.
    """
    energy: NDArray = data.field(0)
    transmission: NDArray = data.field(1)
    low, high = energy_range
    band: NDArray = (energy >= low) & (energy <= high)
    return energy[band], transmission[band]

def interp(
    x: NDArray,
    y: NDArray,
    energy: NDArray,
) -> NDArray:
    """Interpolates `y` values in given `energy` values."""
    # ...
    # some preprocess (?)
    # ...
    return griddata(x, y, energy, method='linear')

def integrate(arr: NDArray, bins: NDArray) -> float:
    """Integrates input array."""
    return np.cumsum(arr * bins)[-1]

In [3]:
def _transform(theta_x: float, theta_y: float) -> float:
    """
    Computes transformation between local-frame and polar
    frame. Input angles are in [deg]. The output is the
    tangent of the polar angle wrt the XY plane.
    """
    tan_x, tan_y = map(
        lambda x: np.tan(np.deg2rad(x)), (theta_x, theta_y),
    )
    return np.sqrt(tan_x ** 2 + tan_y ** 2)

def compute_theta(theta_x: float, theta_y: float) -> float:
    """
    Computes the polar angular coord wrt to the xy plane.
    Both `theta_x` and `theta_y` are in [deg].
    Output angle value is in [deg].
    """
    xi: float = _transform(theta_x, theta_y)
    return np.rad2deg(np.atan(xi))

def project_absrp_correction(
    distance: float,
    theta_x: float,
    theta_y: float,
) -> NDArray:
    """
    Computes the source local-frame coords correction for the
    detector absorption photons distance in a given energy band.
    """
    xi: float = _transform(theta_x, theta_y)
    theta_plane_proj: float = np.sin(np.atan(xi))
    local_angles: NDArray = np.deg2rad(np.array([theta_x, theta_y]))
    phi_local_proj: NDArray = np.tan(local_angles) / (xi + 1e-8)
    return distance * theta_plane_proj * phi_local_proj

- We want to compute the median absorption distance on the detector for an incident photon with energy $E$, for the SDDs inside each LEM-X camera.

- The travel distance $x$ for a photon with energy $E$ is linked to the detector material transmission (Si-based):

$$ T(x, E) \propto \text{exp}[- \rho\mu(E) \cdot x] $$

- The median distance is then:

$$ \hat{\text{x}}(E) = \frac{\text{ln}(2)}{\rho\mu(E)} $$

In [4]:
#settings_path: str = "/mnt/dbb8f47e-da06-47bf-8ef5-038092af70f7/Edos_Magnificent_Manor/PhD_AASS/Coding/IROS_Data/Simulations/camera_settings"
settings_path: str = "/mnt/d/PhD_AASS/Coding/Images_fits/camera_settings"

detSi_matten: FITS_rec = load_fits_data(Path(settings_path, "detectorSi_absrp.fits"))

In [5]:
# extract detector density, thickness and mass attenuation (ON-AXIS)
rho: float = 2.33   # [g/cm3] (from FITS header)
d: float = 0.045    # [cm]
energy, matten = extract_transmission(detSi_matten)  # [keV], [cm2/g]

In [6]:
def compute_median_dist(ph_energy: float) -> float:
    """
    Computes detector median absorption distance
    in [cm] for a photon with `ph_energy` keV.
    """
    mu = interp(energy, matten, ph_energy)
    return np.log(2) / (rho * mu)


ph_energy = 17.0  # [keV]
theta_x, theta_y = 10.0, 10.0

# LoS mean absrp distance in the detector [mm]
dist: float = 10 * compute_median_dist(ph_energy)

_theta = np.deg2rad(compute_theta(theta_x, theta_y))
depth = dist * np.cos(_theta)
print(
    f'Photon mean distance absorption: {dist:.4f} mm\n'
    f'Detector penetration: {depth:.4f} mm (detected: {depth < 10 * d})\n'
    f'Impact deviation: {dist * np.sin(_theta):.4f} mm\n'
)

dsx, dsy = project_absrp_correction(dist, theta_x, theta_y)
print(f'Local-frame coords correction (x, y): {dsx:.4f}, {dsy:.4f} mm')

Photon mean distance absorption: 0.4428 mm
Detector penetration: 0.4296 mm (detected: True)
Impact deviation: 0.1071 mm

Local-frame coords correction (x, y): 0.0758, 0.0758 mm


- If we want to know the median distance in a given energy interval $[E_{1}, E_{2}]$, we have to integrate.

In [8]:
theta_x, theta_y = 10.0, 0.0                      # [deg]
energy_range: tuple[float, float] = (8.0, 14.0)  # [keV]

energy_, matten_ = extract_transmission(detSi_matten, energy_range)  # [keV], [cm2/g]
dist: float = 10 * (np.log(2) / rho) * integrate(1.0 / matten_[:-1], np.diff(energy_)) # [mm]

_theta = np.deg2rad(compute_theta(theta_x, theta_y))
depth = dist * np.cos(_theta)
print(
    f'Photon mean distance absorption: {dist:.4f} mm\n'
    f'Detector penetration: {depth:.4f} mm (detected: {depth < 10 * d})\n'
    f'Impact deviation: {dist * np.sin(_theta):.4f} mm\n'
)

dsx, dsy = project_absrp_correction(dist, theta_x, theta_y)
print(f'Local-frame coords correction (x, y): {dsx:.4f}, {dsy:.4f} mm')

Photon mean distance absorption: 0.7568 mm
Detector penetration: 0.7453 mm (detected: False)
Impact deviation: 0.1314 mm

Local-frame coords correction (x, y): 0.1314, 0.0000 mm
